In [ ]:
class LLMTestAgent(BaseIncidentAgent):
    def test_call(self, prompt: str) -> Dict[str, Any]:
        # Use a very small fallback to ensure LLM is actually called if enabled
        return self.llm_json("Return only JSON with a 'response' key.", prompt, {"response": "LLM disabled or failed."})

# Instantiate the test agent
test_agent = LLMTestAgent(LLM_SETTINGS, enabled=True)

# Make a small test call if the LLM is available
if test_agent.enabled:
    print("Attempting a test LLM call...")
    test_output = test_agent.test_call("Say hello in a JSON object with key 'response'.")
    print("Test LLM output:", test_output)
    if "hello" in str(test_output).lower():
        print("LLM call successful!")
    else:
        print("LLM call returned unexpected output.")
else:
    print("LLM is not enabled or API key is missing/invalid, skipping test call.")

API Key Found. We are good to go...
Attempting a test LLM call...
LLM fallback due to: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-or-v1*************************************************************4f56. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}
Test LLM output: {'response': 'LLM disabled or failed.'}
LLM call returned unexpected output.


In [31]:
class LLMTestAgent(BaseIncidentAgent):
    def test_call(self, prompt: str) -> Dict[str, Any]:
        # Use a very small fallback to ensure LLM is actually called if enabled
        return self.llm_json("Return only JSON with a 'response' key.", prompt, {"response": "LLM disabled or failed."})

# Instantiate the test agent
test_agent = LLMTestAgent(LLM_SETTINGS, enabled=True)

# Make a small test call if the LLM is available
if test_agent.enabled:
    print("Attempting a test LLM call...")
    test_output = test_agent.test_call("Say hello in a JSON object with key 'response'.")
    print("Test LLM output:", test_output)
    if "hello" in str(test_output).lower():
        print("LLM call successful!")
    else:
        print("LLM call returned unexpected output.")
else:
    print("LLM is not enabled or API key is missing/invalid, skipping test call.")

API Key Found. We are good to go...
Attempting a test LLM call...
LLM fallback due to: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-or-v1*************************************************************4f56. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}
Test LLM output: {'response': 'LLM disabled or failed.'}
LLM call returned unexpected output.


# Open This Notebook in Colab

<a href="https://colab.research.google.com/github/dreameraiquest/IncidentIQ/blob/main/notebooks/Multi_Agent_DevOps_Incident_Analysis_Suite_SRC_Aligned_Colab_v3.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


# Multi-Agent DevOps Incident Analysis Suite — SRC-Aligned Colab v3

This notebook is aligned to `src_README.md` and is intentionally split into class-oriented cells that map directly to the future `src/` Python project.

Runtime flow:

```text
config -> models -> utils -> ingest -> signals -> clustering -> rag -> agents -> graph -> reporting -> evals
```

Design rules used here:

- Runtime reads only raw operational logs, never `ground_truth_eval_only`.
- Ground truth is loaded only after inference by `evals/`.
- Deterministic Python extracts evidence, signals, clusters, reports, and eval metrics.
- OpenRouter/OpenAI-compatible LLM calls are used only by agents when a key is configured.
- Fallback deterministic mode remains available for fast, reproducible runs.
- Slack/JIRA actions are preview-only and gated by human approval status.


## v3 finalization focus

This version adds a Colab batch execution layer above the `src`-aligned implementation:

```text
Colab Batch Layer
  -> Batch input discovery
  -> Checkpointed dataset runs
  -> Per-run output directories
  -> Deterministic preprocessing first
  -> Selective LLM/LangGraph execution
  -> Eval only after inference
```

Design intent: every implementation cell still maps cleanly to a future `src/` module, while the top-level batch layer acts as the temporary Colab app/controller.


## 0. Install dependencies

Run this first in Google Colab. The notebook uses `langgraph` if available, but also has a deterministic fallback runner so the pipeline remains runnable cell by cell.


In [1]:
!pip -q install pandas numpy pydantic tqdm scikit-learn matplotlib
!pip -q install langgraph langchain langchain-openai langchain-core openai


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 8.0 MB/s eta 0:00:00


## 1. `config/` — settings and runtime feature flags

Maps to:

```text
src/config/settings.py
src/config/llm_settings.py
src/config/rag_settings.py
src/config/eval_settings.py
src/config/logging_config.py
```


In [32]:
from __future__ import annotations
import os, re, json, zipfile, shutil, hashlib, textwrap, math, statistics, csv
from pathlib import Path
from datetime import datetime, timezone, timedelta
from typing import Any, Dict, List, Optional, Tuple, Iterable, Set, Literal
from collections import Counter, defaultdict
from enum import Enum
from dataclasses import dataclass, field

import pandas as pd
from pydantic import BaseModel, Field
from tqdm.auto import tqdm
from google.colab import userdata


class AppSettings(BaseModel):
    base_dir: Path = Path(os.getenv("INCIDENTIQ_BASE_DIR", "/content/incidentiq_workspace"))
    upload_dir: Path = Path(os.getenv("INCIDENTIQ_UPLOAD_DIR", "/content/incidentiq_workspace/uploads"))
    extract_dir: Path = Path(os.getenv("INCIDENTIQ_EXTRACT_DIR", "/content/incidentiq_workspace/extracted"))
    output_dir: Path = Path(os.getenv("INCIDENTIQ_OUTPUT_DIR", "/content/incidentiq_workspace/outputs"))
    default_time_window_minutes: int = int(os.getenv("DEFAULT_TIME_WINDOW_MINUTES", "15"))
    max_evidence_lines: int = int(os.getenv("MAX_EVIDENCE_LINES", "24"))
    min_signal_weight: float = float(os.getenv("MIN_SIGNAL_WEIGHT", "1.0"))
    enable_preview_mode: bool = os.getenv("ENABLE_PREVIEW_MODE", "true").lower() == "true"
    enable_eval_mode: bool = os.getenv("ENABLE_EVAL_MODE", "true").lower() == "true"
    enable_rag_mode: bool = os.getenv("ENABLE_RAG_MODE", "true").lower() == "true"
    enable_llm_mode: bool = os.getenv("ENABLE_LLM_MODE", "auto").lower() != "false"

    def prepare(self) -> "AppSettings":
        for p in [self.base_dir, self.upload_dir, self.extract_dir, self.output_dir]:
            p.mkdir(parents=True, exist_ok=True)
        return self

class LLMSettings(BaseModel):
    # OpenRouter in Colab: set OPENROUTER_API_KEY in secrets/env.
    # Also supports OPENAI_API_KEY + OPENAI_BASE_URL for OpenAI-compatible providers.
    api_key: Optional[str] = userdata.get("OPENROUTER_API_KEY") or userdata.get("OPENAI_API_KEY")
    base_url: Optional[str] = os.getenv("OPENAI_BASE_URL") or ("https://openrouter.ai/api/v1" if userdata.get("OPENROUTER_API_KEY") else None)
    model_name: str = os.getenv("MODEL_NAME", "openai/gpt-4o-mini")
    temperature: float = float(os.getenv("LLM_TEMPERATURE", "0.0"))
    max_tokens: int = int(os.getenv("LLM_MAX_TOKENS", "900"))

    @property
    def available(self) -> bool:
        print("API Key Found. We are good to go...")
        return bool(self.api_key)

APP_SETTINGS = AppSettings().prepare()
LLM_SETTINGS = LLMSettings()
print("Settings ready. LLM available:", LLM_SETTINGS.available, "base_url:", LLM_SETTINGS.base_url)

API Key Found. We are good to go...
Settings ready. LLM available: True base_url: https://openrouter.ai/api/v1


## 2. `models/` — typed contracts shared across components

Maps to:

```text
src/models/enums.py
src/models/log_event.py
src/models/signal.py
src/models/cluster.py
src/models/incident.py
src/models/rca.py
src/models/remediation.py
src/models/timeline.py
src/models/runbook.py
src/models/actions.py
src/models/reports.py
src/models/evals.py
src/models/graph_state.py
```


In [17]:
class IncidentCategory(str, Enum):
    DATABASE = "Database"
    NETWORK = "Network"
    AUTHENTICATION = "Authentication"
    MEMORY_CPU = "Memory/CPU"
    DEPLOYMENT_REGRESSION = "Deployment regression"
    API_TIMEOUT = "API timeout"
    QUEUE_BACKLOG = "Queue backlog"
    DISK_STORAGE = "Disk/storage"
    EXTERNAL_DEPENDENCY = "External dependency"
    UNKNOWN = "Unknown"

class SeverityLevel(str, Enum):
    P1 = "P1"
    P2 = "P2"
    P3 = "P3"
    P4 = "P4"
    UNKNOWN = "UNKNOWN"

class SignalType(str, Enum):
    ERROR_PATTERN = "error_pattern"
    LATENCY = "latency"
    RESOURCE = "resource"
    DEPLOYMENT = "deployment"
    AUTH = "auth"
    STORAGE = "storage"
    QUEUE = "queue"
    NETWORK = "network"
    UNKNOWN = "unknown"

class ApprovalStatus(str, Enum):
    PREVIEW_ONLY = "PREVIEW_ONLY"
    PENDING_HUMAN_APPROVAL = "PENDING_HUMAN_APPROVAL"
    APPROVED = "APPROVED"
    REJECTED = "REJECTED"

class RawLogLine(BaseModel):
    source_file: str
    line_no: int
    raw_line: str

class LogEvent(BaseModel):
    source_file: str
    line_no: int
    timestamp: Optional[datetime] = None
    level: str = "UNKNOWN"
    service: Optional[str] = None
    message: str
    trace_id: Optional[str] = None
    span_id: Optional[str] = None
    host: Optional[str] = None
    namespace: Optional[str] = None
    pod: Optional[str] = None
    container: Optional[str] = None
    raw_line: str
    parse_status: str = "parsed"
    metadata: Dict[str, Any] = Field(default_factory=dict)

class SignalRule(BaseModel):
    rule_id: str
    category: IncidentCategory
    signal_type: SignalType
    patterns: List[str]
    weight: float
    severity_hint: SeverityLevel = SeverityLevel.UNKNOWN
    description: str

class SignalEvidence(BaseModel):
    source_file: str
    line_no: int
    timestamp: Optional[datetime] = None
    service: Optional[str] = None
    level: str = "UNKNOWN"
    message: str
    trace_id: Optional[str] = None

class SignalMatch(BaseModel):
    rule_id: str
    category: IncidentCategory
    signal_type: SignalType
    weight: float
    severity_hint: SeverityLevel
    evidence: SignalEvidence
    matched_text: str

class EvidenceCluster(BaseModel):
    cluster_id: str
    candidate_category: IncidentCategory
    signature: str
    affected_services: List[str]
    start_ts: Optional[datetime] = None
    end_ts: Optional[datetime] = None
    signal_count: int
    total_weight: float
    severity_hint: SeverityLevel
    evidence: List[SignalEvidence]
    rule_ids: List[str]
    trace_ids: List[str] = Field(default_factory=list)
    summary: str = ""

class TimelineEvent(BaseModel):
    timestamp: Optional[datetime]
    source_file: str
    line_no: int
    service: Optional[str]
    level: str
    message: str

class IncidentTimeline(BaseModel):
    events: List[TimelineEvent]
    interpretation: str

class RootCauseAnalysis(BaseModel):
    primary_root_cause: str
    alternative_causes: List[str]
    supporting_evidence: List[str]
    missing_evidence: List[str]
    confidence: float

class RemediationPlan(BaseModel):
    immediate_actions: List[str]
    validation_checks: List[str]
    safety_notes: List[str]
    requires_human_approval: bool = True

class NotificationPayload(BaseModel):
    channel: str = "preview"
    title: str
    body: str
    approval_status: ApprovalStatus = ApprovalStatus.PREVIEW_ONLY

class TicketPayload(BaseModel):
    system: str = "preview"
    title: str
    priority: SeverityLevel
    labels: List[str]
    description: str
    approval_status: ApprovalStatus = ApprovalStatus.PREVIEW_ONLY

class CookbookChecklist(BaseModel):
    title: str
    steps: List[str]
    validation: List[str]
    safety_notes: List[str]

class RunbookChunk(BaseModel):
    service: str = "generic"
    incident_type: str
    symptoms: List[str]
    diagnostics: List[str]
    remediation: List[str]
    validation: List[str]
    safety_notes: List[str]
    owner: str = "sre"
    source: str = "embedded_runbook"

class RunbookRetrievalResult(BaseModel):
    chunks: List[RunbookChunk]
    query: str

class IncidentAnalysisResult(BaseModel):
    incident_id: str
    cluster: EvidenceCluster
    category: IncidentCategory
    severity: SeverityLevel
    confidence: float
    symptom_vs_cause: str
    timeline: IncidentTimeline
    rca: RootCauseAnalysis
    remediation: RemediationPlan
    notification: NotificationPayload
    ticket: TicketPayload
    cookbook: CookbookChecklist
    critic_findings: List[str]
    evidence_grounded: bool = True

class ExportManifest(BaseModel):
    markdown_report: Optional[str] = None
    json_export: Optional[str] = None
    csv_export: Optional[str] = None
    output_zip: Optional[str] = None

class GroundTruthCase(BaseModel):
    case_id: str
    raw_file: str
    expected_category: str
    expected_severity: str
    affected_services: List[str] = Field(default_factory=list)
    expected_evidence_terms: List[Any] = Field(default_factory=list)
    expected_behavior: Dict[str, Any] = Field(default_factory=dict)

class EvalResult(BaseModel):
    case_id: str
    matched_incident_id: Optional[str]
    category_match: bool
    severity_match: bool
    ticket_match: bool
    evidence_term_hit: bool

class EvalSummary(BaseModel):
    total_cases: int
    matched_cases: int
    category_recall: float
    severity_accuracy_on_matched: float
    ticket_trigger_accuracy_on_matched: float
    evidence_term_hit_rate_on_matched: float
    results: List[EvalResult]

class IncidentGraphState(BaseModel):
    input_paths: List[str] = Field(default_factory=list)
    runtime_root: Optional[str] = None
    raw_files: List[str] = Field(default_factory=list)
    ground_truth_files: List[str] = Field(default_factory=list)
    events: List[LogEvent] = Field(default_factory=list)
    signals: List[SignalMatch] = Field(default_factory=list)
    clusters: List[EvidenceCluster] = Field(default_factory=list)
    incidents: List[IncidentAnalysisResult] = Field(default_factory=list)
    eval_summary: Optional[EvalSummary] = None
    exports: ExportManifest = Field(default_factory=ExportManifest)
    errors: List[str] = Field(default_factory=list)


## 3. `utils/` — safe paths, timestamps, text, JSON, hashes

Maps to:

```text
src/utils/file_utils.py
src/utils/time_utils.py
src/utils/json_utils.py
src/utils/text_utils.py
src/utils/hash_utils.py
src/utils/errors.py
```


In [18]:
def stable_hash(text: str, length: int = 12) -> str:
    return hashlib.sha256(text.encode("utf-8", errors="ignore")).hexdigest()[:length]

def safe_relpath(path: Path, root: Path) -> str:
    try:
        return str(path.resolve().relative_to(root.resolve())).replace("\\", "/")
    except Exception:
        return str(path).replace("\\", "/")

def parse_timestamp(value: Any) -> Optional[datetime]:
    if not value:
        return None
    if isinstance(value, datetime):
        return value
    s = str(value).strip()
    s = s.replace("Z", "+00:00")
    for fmt in [None, "%Y-%m-%d %H:%M:%S,%f", "%Y-%m-%d %H:%M:%S", "%b %d %H:%M:%S"]:
        try:
            if fmt is None:
                return datetime.fromisoformat(s)
            dt = datetime.strptime(s, fmt)
            if fmt == "%b %d %H:%M:%S":
                dt = dt.replace(year=datetime.now().year)
            return dt
        except Exception:
            pass
    return None

def normalize_level(value: Any, message: str = "") -> str:
    s = (str(value or "") + " " + message[:100]).upper()
    for lvl in ["FATAL", "CRITICAL", "ERROR", "WARN", "WARNING", "INFO", "DEBUG", "TRACE"]:
        if lvl in s:
            return "WARN" if lvl == "WARNING" else lvl
    return "UNKNOWN"

def compact_message(text: str, limit: int = 220) -> str:
    text = re.sub(r"\s+", " ", str(text or "")).strip()
    return text if len(text) <= limit else text[: limit - 3] + "..."

def flatten_terms(obj: Any) -> List[str]:
    if obj is None:
        return []
    if isinstance(obj, str):
        return [obj]
    if isinstance(obj, list):
        out = []
        for x in obj:
            out.extend(flatten_terms(x))
        return out
    return [str(obj)]

def evidence_ref(ev: SignalEvidence) -> str:
    return f"{ev.source_file}:{ev.line_no} - {compact_message(ev.message, 180)}"


## 4. `ingest/` — file loading, ZIP extraction, discovery, parsing, normalization

Maps to:

```text
src/ingest/input_validator.py
src/ingest/zip_loader.py
src/ingest/log_discovery.py
src/ingest/raw_log_reader.py
src/ingest/log_parser.py
src/ingest/normalizer.py
```


In [19]:
class InputValidator:
    allowed_suffixes = {".zip", ".jsonl", ".log", ".txt"}
    def validate(self, input_paths: List[str]) -> List[Path]:
        paths = [Path(p) for p in input_paths]
        bad = [str(p) for p in paths if not p.exists() or p.suffix.lower() not in self.allowed_suffixes]
        if bad:
            raise ValueError(f"Invalid input files: {bad}")
        return paths

class SafeZipLoader:
    def extract(self, zip_path: Path, extract_dir: Path) -> Path:
        target = extract_dir / zip_path.stem
        if target.exists():
            shutil.rmtree(target)
        target.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path) as zf:
            for member in zf.infolist():
                out = target / member.filename
                if not str(out.resolve()).startswith(str(target.resolve())):
                    raise ValueError(f"Unsafe zip member path: {member.filename}")
            zf.extractall(target)
        return target

class FileLoader:
    def __init__(self, settings: AppSettings):
        self.settings = settings
        self.validator = InputValidator()
        self.zip_loader = SafeZipLoader()
    def load(self, input_paths: List[str]) -> Path:
        paths = self.validator.validate(input_paths)
        run_root = self.settings.extract_dir / f"run_{stable_hash('|'.join(map(str, paths)) + str(datetime.now()))}"
        if run_root.exists(): shutil.rmtree(run_root)
        run_root.mkdir(parents=True, exist_ok=True)
        for p in paths:
            if p.suffix.lower() == ".zip":
                extracted = self.zip_loader.extract(p, run_root)
                print(f"Extracted {p.name} -> {extracted}")
            else:
                dest = run_root / "raw_logs" / p.name
                dest.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(p, dest)
        return run_root

class LogFileDiscovery:
    runtime_suffixes = {".jsonl", ".log", ".txt"}
    def discover_runtime_logs(self, root: Path) -> List[Path]:
        files = []
        for p in root.rglob("*"):
            rel = str(p.relative_to(root)).replace("\\", "/")
            if p.is_file() and p.suffix.lower() in self.runtime_suffixes and "ground_truth_eval_only" not in rel:
                files.append(p)
        return sorted(files)
    def discover_ground_truth(self, root: Path) -> List[Path]:
        return sorted([p for p in root.rglob("ground_truth_eval_only/*") if p.is_file() and p.suffix.lower() in {".json", ".csv"}])

class RawLogReader:
    def read_lines(self, path: Path, root: Path) -> List[RawLogLine]:
        rel = safe_relpath(path, root)
        out=[]
        with open(path, "r", encoding="utf-8", errors="replace") as f:
            for i, line in enumerate(f, 1):
                if line.strip():
                    out.append(RawLogLine(source_file=rel, line_no=i, raw_line=line.rstrip("\n")))
        return out

class LogParser:
    ts_keys = ["timestamp", "ts", "time", "@timestamp", "datetime"]
    service_keys = ["service", "app", "application", "logger", "component", "container", "pod"]
    msg_keys = ["message", "msg", "log", "event", "error", "detail"]
    trace_keys = ["trace_id", "traceId", "trace", "correlation_id", "correlationId", "request_id", "requestId"]
    def parse_line(self, raw: RawLogLine) -> LogEvent:
        line = raw.raw_line.strip()
        data = None
        if line.startswith("{"):
            try: data = json.loads(line)
            except Exception: data = None
        if isinstance(data, dict):
            msg = next((str(data.get(k)) for k in self.msg_keys if data.get(k) is not None), line)
            ts = next((data.get(k) for k in self.ts_keys if data.get(k) is not None), None)
            service = next((str(data.get(k)) for k in self.service_keys if data.get(k) is not None), None)
            trace = next((str(data.get(k)) for k in self.trace_keys if data.get(k) is not None), None)
            return LogEvent(
                source_file=raw.source_file, line_no=raw.line_no, timestamp=parse_timestamp(ts),
                level=normalize_level(data.get("level") or data.get("severity"), msg), service=service,
                message=msg, trace_id=trace, span_id=data.get("span_id") or data.get("spanId"),
                host=data.get("host"), namespace=data.get("namespace"), pod=data.get("pod"), container=data.get("container"),
                raw_line=raw.raw_line, parse_status="parsed_json", metadata={k:v for k,v in data.items() if k not in self.msg_keys}
            )
        # Plain-text heuristic parser.
        ts_match = re.search(r"(\d{4}-\d{2}-\d{2}[T ]\d{2}:\d{2}:\d{2}(?:\.\d+)?Z?)", line)
        level = normalize_level(None, line)
        svc_match = re.search(r"(?:service|app|component|logger|pod)=([\w.-]+)", line, re.I)
        trace_match = re.search(r"(?:trace_id|traceId|correlation_id|request_id)=([\w.-]+)", line, re.I)
        return LogEvent(
            source_file=raw.source_file, line_no=raw.line_no, timestamp=parse_timestamp(ts_match.group(1) if ts_match else None),
            level=level, service=svc_match.group(1) if svc_match else None, message=line,
            trace_id=trace_match.group(1) if trace_match else None, raw_line=raw.raw_line,
            parse_status="parsed_text" if ts_match or svc_match else "parsed_loose", metadata={}
        )

class LogEventNormalizer:
    def normalize(self, event: LogEvent) -> LogEvent:
        if not event.service:
            # infer common service-like tokens from message/file without relying on category folders
            m = re.search(r"\b([a-z][a-z0-9-]+(?:-api|-service|-worker|-consumer|-gateway|primary|broker|dns|kubelet|agent))\b", event.message)
            event.service = m.group(1) if m else None
        event.level = normalize_level(event.level, event.message)
        event.message = compact_message(event.message, 1000)
        return event

class IngestionService:
    def __init__(self, settings: AppSettings):
        self.loader = FileLoader(settings); self.discovery = LogFileDiscovery(); self.reader = RawLogReader(); self.parser = LogParser(); self.normalizer = LogEventNormalizer()
    def run(self, input_paths: List[str]) -> Tuple[Path, List[Path], List[Path], List[LogEvent]]:
        root = self.loader.load(input_paths)
        raw_files = self.discovery.discover_runtime_logs(root)
        gt_files = self.discovery.discover_ground_truth(root)
        events=[]
        for f in tqdm(raw_files, desc="Parsing logs"):
            for raw in self.reader.read_lines(f, root):
                events.append(self.normalizer.normalize(self.parser.parse_line(raw)))
        return root, raw_files, gt_files, events


## 5. `signals/` — deterministic signal rules and severity/category heuristics

Maps to:

```text
src/signals/rules.py
src/signals/pattern_library.py
src/signals/rule_engine.py
src/signals/severity_rules.py
src/signals/category_rules.py
src/signals/signal_extractor.py
src/signals/signal_weights.py
```


In [20]:
def rule(rule_id, category, signal_type, patterns, weight, severity, description):
    return SignalRule(rule_id=rule_id, category=category, signal_type=signal_type, patterns=patterns, weight=weight, severity_hint=severity, description=description)

SIGNAL_RULES: List[SignalRule] = [
    rule("DB_POOL", IncidentCategory.DATABASE, SignalType.ERROR_PATTERN, [r"HikariPool.*Connection is not available", r"remaining connection slots", r"timeout acquiring JDBC connection", r"too many clients"], 5, SeverityLevel.P1, "DB connection pool exhaustion"),
    rule("DB_DEADLOCK", IncidentCategory.DATABASE, SignalType.ERROR_PATTERN, [r"deadlock detected", r"ShareLock on transaction", r"could not serialize access"], 4, SeverityLevel.P2, "DB deadlock/lock contention"),
    rule("DB_SLOW_QUERY", IncidentCategory.DATABASE, SignalType.LATENCY, [r"slow query", r"sequential scan", r"work_mem", r"query exceeded", r"statement timeout"], 3, SeverityLevel.P2, "DB slow query/resource spill"),
    rule("NET_RESET", IncidentCategory.NETWORK, SignalType.NETWORK, [r"connection reset by peer", r"upstream connection refused", r"ECONNRESET", r"TCP retransmits"], 4, SeverityLevel.P2, "Network reset/refusal"),
    rule("NET_DNS", IncidentCategory.NETWORK, SignalType.NETWORK, [r"DNS failure", r"Name or service not known", r"SERVFAIL", r"kube-dns", r"NXDOMAIN"], 4, SeverityLevel.P2, "DNS instability"),
    rule("NET_TLS", IncidentCategory.NETWORK, SignalType.NETWORK, [r"TLS handshake timeout", r"certificate verify failed", r"SSL", r"handshake failure"], 3, SeverityLevel.P2, "TLS/network handshake issue"),
    rule("AUTH_JWK", IncidentCategory.AUTHENTICATION, SignalType.AUTH, [r"JWK refresh failed", r"jwks.*refresh", r"key rotation"], 4, SeverityLevel.P2, "JWK/key-refresh authentication issue"),
    rule("AUTH_RBAC", IncidentCategory.AUTHENTICATION, SignalType.AUTH, [r"RBAC deny", r"permission denied", r"403", r"forbidden", r"policy denied"], 2, SeverityLevel.P3, "RBAC/authorization issue"),
    rule("AUTH_JWT", IncidentCategory.AUTHENTICATION, SignalType.AUTH, [r"JWT.*issuer mismatch", r"invalid signature", r"token validation failed", r"invalid token signature", r"401"], 4, SeverityLevel.P2, "JWT validation issue"),
    rule("MEM_OOM", IncidentCategory.MEMORY_CPU, SignalType.RESOURCE, [r"OOMKilled", r"out of memory", r"heap high", r"memory limit", r"restart loop"], 5, SeverityLevel.P1, "Memory pressure/OOM"),
    rule("CPU_THROTTLE", IncidentCategory.MEMORY_CPU, SignalType.RESOURCE, [r"CPU throttling", r"cpu saturation", r"high CPU"], 3, SeverityLevel.P2, "CPU throttling/saturation"),
    rule("GC_PAUSE", IncidentCategory.MEMORY_CPU, SignalType.RESOURCE, [r"GC pause", r"stop-the-world", r"garbage collection"], 3, SeverityLevel.P2, "GC pause latency"),
    rule("DEPLOY_ERROR", IncidentCategory.DEPLOYMENT_REGRESSION, SignalType.DEPLOYMENT, [r"canary failed", r"rollback recommended", r"post-deploy.*error", r"new version.*error", r"rollout.*failed"], 5, SeverityLevel.P1, "High-impact deployment regression"),
    rule("DEPLOY_FEATURE", IncidentCategory.DEPLOYMENT_REGRESSION, SignalType.DEPLOYMENT, [r"feature flag enabled", r"flag.*enabled", r"feature.*caused"], 3, SeverityLevel.P2, "Feature flag regression"),
    rule("DEPLOY_CONFIG", IncidentCategory.DEPLOYMENT_REGRESSION, SignalType.DEPLOYMENT, [r"configmap changed", r"config.*reload", r"bad config", r"configuration regression", r"rollout started", r"new version"], 3, SeverityLevel.P2, "Deployment/config regression"),
    rule("API_504", IncidentCategory.API_TIMEOUT, SignalType.LATENCY, [r"HTTP 504", r"504 Gateway Timeout", r"gateway timeout", r"upstream timed out"], 4, SeverityLevel.P2, "API timeout"),
    rule("API_P95", IncidentCategory.API_TIMEOUT, SignalType.LATENCY, [r"p95 latency", r"latency breached", r"request timed out"], 3, SeverityLevel.P2, "API latency SLO breach"),
    rule("API_CIRCUIT", IncidentCategory.API_TIMEOUT, SignalType.LATENCY, [r"circuit breaker opened", r"circuit open"], 3, SeverityLevel.P2, "Circuit breaker opened"),
    rule("QUEUE_LAG", IncidentCategory.QUEUE_BACKLOG, SignalType.QUEUE, [r"consumer lag", r"queue backlog", r"lag by partition"], 4, SeverityLevel.P2, "Queue consumer lag"),
    rule("QUEUE_POISON", IncidentCategory.QUEUE_BACKLOG, SignalType.QUEUE, [r"DLQ", r"dead letter", r"retry topic", r"poison message"], 4, SeverityLevel.P2, "Poison message/DLQ growth"),
    rule("QUEUE_REBALANCE", IncidentCategory.QUEUE_BACKLOG, SignalType.QUEUE, [r"rebalance loop", r"consumer group rebalanc", r"partition revoked"], 2, SeverityLevel.P3, "Queue rebalance instability"),
    rule("DISK_FULL", IncidentCategory.DISK_STORAGE, SignalType.STORAGE, [r"No space left on device", r"disk pressure", r"PVC.*full", r"volume full"], 5, SeverityLevel.P1, "Disk/storage exhaustion"),
    rule("DISK_ROTATE", IncidentCategory.DISK_STORAGE, SignalType.STORAGE, [r"logrotate failed", r"log rotation", r"large log file"], 3, SeverityLevel.P2, "Log rotation/disk growth issue"),
    rule("STORAGE_IO", IncidentCategory.DISK_STORAGE, SignalType.STORAGE, [r"IO wait high", r"persistent volume latency", r"storage latency", r"iowait"], 3, SeverityLevel.P2, "Storage IO latency"),
    rule("EXT_RATE", IncidentCategory.EXTERNAL_DEPENDENCY, SignalType.ERROR_PATTERN, [r"HTTP 429", r"Retry-After", r"rate limit", r"throttl"], 2, SeverityLevel.P3, "External dependency rate limiting"),
    rule("EXT_MD", IncidentCategory.EXTERNAL_DEPENDENCY, SignalType.ERROR_PATTERN, [r"feed delayed", r"market data", r"vendor degraded"], 3, SeverityLevel.P2, "External data/vendor degradation"),
    rule("EXT_PAY", IncidentCategory.EXTERNAL_DEPENDENCY, SignalType.ERROR_PATTERN, [r"provider 5xx", r"vendor latency", r"third-party", r"payment provider", r"vendor.*5xx"], 3, SeverityLevel.P2, "External provider degradation"),
    rule("UNKNOWN_INCOMPLETE", IncidentCategory.UNKNOWN, SignalType.UNKNOWN, [r"missing correlation ID", r"incomplete trace", r"unexpected state transition", r"redacted stack trace", r"insufficient evidence", r"unknown error"], 2, SeverityLevel.P3, "Unknown/incomplete evidence"),
]

class SignalRuleEngine:
    def __init__(self, rules: List[SignalRule]):
        self.rules = rules
        self.compiled = [(r, [re.compile(p, re.I) for p in r.patterns]) for r in rules]
    def match_event(self, event: LogEvent) -> List[SignalMatch]:
        text = f"{event.level} {event.service or ''} {event.message} {event.raw_line}"
        matches=[]
        for r, pats in self.compiled:
            for pat in pats:
                m = pat.search(text)
                if m:
                    ev = SignalEvidence(source_file=event.source_file, line_no=event.line_no, timestamp=event.timestamp, service=event.service, level=event.level, message=event.message, trace_id=event.trace_id)
                    matches.append(SignalMatch(rule_id=r.rule_id, category=r.category, signal_type=r.signal_type, weight=r.weight, severity_hint=r.severity_hint, evidence=ev, matched_text=m.group(0)))
                    break
        return matches

class SignalExtractor:
    def __init__(self, engine: SignalRuleEngine): self.engine = engine
    def extract(self, events: List[LogEvent]) -> List[SignalMatch]:
        out=[]
        for e in tqdm(events, desc="Extracting signals"):
            out.extend(self.engine.match_event(e))
        return out

class SeverityHeuristicEngine:
    priority = {SeverityLevel.P1:4, SeverityLevel.P2:3, SeverityLevel.P3:2, SeverityLevel.P4:1, SeverityLevel.UNKNOWN:0}
    def estimate(self, matches: List[SignalMatch]) -> SeverityLevel:
        if not matches: return SeverityLevel.UNKNOWN
        total = sum(m.weight for m in matches)
        max_hint = max((m.severity_hint for m in matches), key=lambda x: self.priority.get(x, 0))
        err_count = sum(1 for m in matches if m.evidence.level in {"FATAL","CRITICAL","ERROR"})
        # Severity should primarily follow the strongest signal class.
        # Counts increase confidence, but should not upgrade intentionally P3-only signals like RBAC, rate limiting, or unknown telemetry gaps.
        if max_hint == SeverityLevel.P1: return SeverityLevel.P1
        if max_hint == SeverityLevel.P2: return SeverityLevel.P2
        if max_hint == SeverityLevel.P3: return SeverityLevel.P3
        if total >= 1: return SeverityLevel.P4
        return SeverityLevel.UNKNOWN


## 6. `clustering/` — hybrid evidence clustering and de-duplication

Maps to:

```text
src/clustering/evidence_clusterer.py
src/clustering/cluster_strategy.py
src/clustering/time_window_clusterer.py
src/clustering/service_clusterer.py
src/clustering/trace_clusterer.py
src/clustering/signature_clusterer.py
src/clustering/deduplicator.py
```


In [21]:
class HybridClusterStrategy:
    def __init__(self, time_window_minutes: int = 15):
        self.time_window_minutes = time_window_minutes
    def time_bucket(self, ts: Optional[datetime]) -> str:
        if not ts: return "no_ts"
        minute = (ts.minute // self.time_window_minutes) * self.time_window_minutes
        return ts.replace(minute=minute, second=0, microsecond=0).isoformat()
    def signature(self, m: SignalMatch) -> str:
        # Keep strong incident signatures separate, while still allowing service/time grouping.
        return m.rule_id
    def key(self, m: SignalMatch) -> Tuple[str,str,str,str]:
        file_key = m.evidence.source_file
        return (file_key, m.category.value, self.signature(m), self.time_bucket(m.evidence.timestamp))

class EvidenceClusterer:
    def __init__(self, severity_engine: SeverityHeuristicEngine, strategy: HybridClusterStrategy, settings: AppSettings):
        self.severity_engine = severity_engine; self.strategy = strategy; self.settings = settings
    def cluster(self, signals: List[SignalMatch]) -> List[EvidenceCluster]:
        grouped=defaultdict(list)
        for s in signals:
            if s.weight >= self.settings.min_signal_weight:
                grouped[self.strategy.key(s)].append(s)
        clusters=[]
        for key, ms in grouped.items():
            file_key, cat, sig, bucket = key
            services = sorted({m.evidence.service for m in ms if m.evidence.service})
            traces = sorted({m.evidence.trace_id for m in ms if m.evidence.trace_id})[:20]
            timestamps = [m.evidence.timestamp for m in ms if m.evidence.timestamp]
            evidence = [m.evidence for m in sorted(ms, key=lambda x: (x.evidence.timestamp or datetime.min, x.evidence.line_no))]
            severity = self.severity_engine.estimate(ms)
            cid = stable_hash("|".join([file_key, cat, sig, bucket]))
            summary = f"{cat} candidate from {sig}: {len(ms)} signal(s), services={services[:4]}"
            clusters.append(EvidenceCluster(
                cluster_id=f"inc_{cid}", candidate_category=IncidentCategory(cat), signature=sig,
                affected_services=services, start_ts=min(timestamps) if timestamps else None, end_ts=max(timestamps) if timestamps else None,
                signal_count=len(ms), total_weight=sum(m.weight for m in ms), severity_hint=severity,
                evidence=evidence[:self.settings.max_evidence_lines], rule_ids=sorted({m.rule_id for m in ms}), trace_ids=traces, summary=summary
            ))
        # Prefer high-confidence clusters, but keep unknown/incomplete evidence.
        return sorted(clusters, key=lambda c: (-c.total_weight, c.candidate_category.value, c.cluster_id))

class IncidentDeduplicator:
    def dedupe(self, clusters: List[EvidenceCluster]) -> List[EvidenceCluster]:
        seen=set(); out=[]
        for c in clusters:
            evidence_key = tuple((e.source_file, e.line_no) for e in c.evidence[:5])
            key=(c.candidate_category.value, c.signature, evidence_key)
            if key not in seen:
                seen.add(key); out.append(c)
        return out


## 7. `rag/` — embedded runbook store and lexical retriever

Maps to:

```text
src/rag/runbook_store.py
src/rag/retriever.py
src/rag/query_builder.py
src/rag/chunker.py
src/rag/reranker.py
```

This is intentionally lexical for speed and reproducibility. It can be upgraded to FAISS/Chroma later without changing agent contracts.


In [22]:
RUNBOOKS = [
    RunbookChunk(incident_type="Database", symptoms=["HikariPool", "deadlock", "slow query", "remaining connection slots"], diagnostics=["Check pg_stat_activity", "Inspect lock waits", "Review DB connection pool metrics"], remediation=["Reduce request pressure", "Tune pool only after DB capacity check", "Terminate clearly idle blockers with approval"], validation=["DB wait time returns to baseline", "5xx rate drops", "Pool acquisition latency normalizes"], safety_notes=["Do not restart DB primary blindly"]),
    RunbookChunk(incident_type="Network", symptoms=["connection reset", "DNS failure", "TLS handshake"], diagnostics=["Check DNS/CoreDNS", "Inspect upstream health", "Validate TLS cert chain"], remediation=["Fail over unhealthy upstream", "Rollback bad network policy", "Refresh certificate/config after approval"], validation=["Error resets drop", "DNS lookup succeeds", "TLS handshakes recover"], safety_notes=["Avoid broad firewall changes without approval"]),
    RunbookChunk(incident_type="Authentication", symptoms=["JWT", "issuer mismatch", "invalid signature", "401", "403"], diagnostics=["Check issuer/audience config", "Validate JWK refresh", "Inspect RBAC policy changes"], remediation=["Rollback auth config", "Refresh JWK cache", "Apply RBAC fix after review"], validation=["401/403 rate normalizes", "Token validation succeeds"], safety_notes=["Do not weaken auth checks to restore traffic"]),
    RunbookChunk(incident_type="Memory/CPU", symptoms=["OOMKilled", "heap", "GC pause", "CPU throttling"], diagnostics=["Check pod restarts", "Inspect memory/CPU limits", "Review heap dump if safe"], remediation=["Scale replicas", "Raise limits only with capacity", "Rollback memory-heavy release"], validation=["Restart loop stops", "GC pause decreases", "Latency recovers"], safety_notes=["Do not delete pods repeatedly without root cause"]),
    RunbookChunk(incident_type="Deployment regression", symptoms=["rollout", "new version", "canary failed", "feature flag"], diagnostics=["Compare pre/post deploy metrics", "Inspect configmap and feature flags", "Review canary error rate"], remediation=["Rollback release", "Disable suspect flag", "Revert config change"], validation=["New-version errors stop", "Canary passes", "SLO recovers"], safety_notes=["Treat correlation as evidence, not proof"]),
    RunbookChunk(incident_type="API timeout", symptoms=["504", "gateway timeout", "upstream timed out", "p95 latency"], diagnostics=["Identify slow upstream", "Check dependency saturation", "Inspect circuit breaker"], remediation=["Reduce timeout cascade", "Scale downstream safely", "Open mitigation ticket"], validation=["p95 latency drops", "504 rate returns to baseline"], safety_notes=["Do not only raise timeouts; find cause"]),
    RunbookChunk(incident_type="Queue backlog", symptoms=["consumer lag", "DLQ", "poison message", "rebalance"], diagnostics=["Check lag by partition", "Inspect DLQ samples", "Find poison messages"], remediation=["Pause poison producer", "Scale consumers", "Move poison messages with approval"], validation=["Lag drains", "DLQ growth stops"], safety_notes=["Do not replay DLQ blindly"]),
    RunbookChunk(incident_type="Disk/storage", symptoms=["No space left", "disk pressure", "logrotate", "IO wait"], diagnostics=["Check filesystem usage", "Find large logs", "Inspect PVC latency"], remediation=["Free safe temp/log files", "Expand volume", "Fix log rotation"], validation=["Disk pressure clears", "Writes succeed", "IO wait drops"], safety_notes=["Do not delete unknown data files"]),
    RunbookChunk(incident_type="External dependency", symptoms=["provider 5xx", "vendor latency", "HTTP 429", "Retry-After"], diagnostics=["Check vendor status", "Inspect retry budget", "Confirm rate limits"], remediation=["Enable degradation mode", "Respect Retry-After", "Throttle callers"], validation=["Vendor errors fall", "Retries stabilize"], safety_notes=["Avoid retry storms"]),
    RunbookChunk(incident_type="Unknown", symptoms=["missing correlation", "incomplete trace", "redacted"], diagnostics=["Request missing traces", "Increase structured logging", "Check telemetry gaps"], remediation=["Open investigation ticket", "Collect more evidence"], validation=["Evidence quality improves"], safety_notes=["Do not take destructive action with insufficient evidence"]),
]

class RetrievalQueryBuilder:
    def build(self, cluster: EvidenceCluster) -> str:
        ev = " ".join(e.message for e in cluster.evidence[:6])
        return f"{cluster.candidate_category.value} {cluster.signature} {' '.join(cluster.affected_services)} {ev}"

class RunbookStore:
    def __init__(self, chunks: List[RunbookChunk]): self.chunks = chunks

class RunbookRetriever:
    def __init__(self, store: RunbookStore): self.store = store; self.query_builder = RetrievalQueryBuilder()
    def retrieve(self, cluster: EvidenceCluster, top_k: int = 3) -> RunbookRetrievalResult:
        query = self.query_builder.build(cluster).lower()
        scored=[]
        for c in self.store.chunks:
            hay = " ".join([c.incident_type] + c.symptoms + c.diagnostics + c.remediation).lower()
            score = sum(1 for tok in set(re.findall(r"[a-z0-9]+", query)) if tok in hay)
            if c.incident_type.lower() == cluster.candidate_category.value.lower(): score += 10
            scored.append((score, c))
        return RunbookRetrievalResult(query=query, chunks=[c for s,c in sorted(scored, key=lambda x:-x[0])[:top_k] if s > 0])


## 8. `agents/` — focused agents with structured outputs

Maps to:

```text
src/agents/base_agent.py
src/agents/classifier_agent.py
src/agents/ambiguity_agent.py
src/agents/symptom_cause_agent.py
src/agents/timeline_agent.py
src/agents/rca_agent.py
src/agents/remediation_agent.py
src/agents/critic_agent.py
src/agents/notification_agent.py
src/agents/ticket_agent.py
src/agents/cookbook_agent.py
src/agents/human_approval_agent.py
```

Agents do not read files directly, do not access hidden ground truth, and cite evidence by file/line.


In [23]:
class BaseIncidentAgent:
    def __init__(self, llm_settings: LLMSettings, enabled: bool = True):
        self.llm_settings = llm_settings; self.enabled = enabled and llm_settings.available
        self.client = None
        if self.enabled:
            try:
                from openai import OpenAI
                self.client = OpenAI(api_key=llm_settings.api_key, base_url=llm_settings.base_url)
            except Exception as e:
                print("LLM client unavailable; using deterministic fallback:", e)
                self.enabled = False
    def llm_json(self, system: str, user: str, fallback: Dict[str, Any]) -> Dict[str, Any]:
        if not self.enabled or not self.client:
            return fallback
        try:
            resp = self.client.chat.completions.create(
                model=self.llm_settings.model_name,
                temperature=self.llm_settings.temperature,
                max_tokens=self.llm_settings.max_tokens,
                messages=[{"role":"system","content":system},{"role":"user","content":user}],
                response_format={"type":"json_object"},
            )
            return json.loads(resp.choices[0].message.content)
        except Exception as e:
            print("LLM fallback due to:", e)
            return fallback

class IncidentClassifierAgent(BaseIncidentAgent):
    def run(self, cluster: EvidenceCluster, runbooks: RunbookRetrievalResult) -> Dict[str, Any]:
        fallback = {
            "category": cluster.candidate_category.value,
            "severity": cluster.severity_hint.value,
            "confidence": min(0.95, 0.55 + cluster.total_weight / 20),
            "symptom_vs_cause": f"Primary candidate is {cluster.candidate_category.value}; related timeout/error lines may be symptoms unless their own signature dominates. Signature={cluster.signature}."
        }
        evidence = "\n".join(evidence_ref(e) for e in cluster.evidence[:12])
        rb = "\n".join(c.incident_type + ": " + "; ".join(c.diagnostics[:2]) for c in runbooks.chunks)
        return self.llm_json("Return only JSON with category,severity,confidence,symptom_vs_cause. Be evidence-grounded.", f"Cluster: {cluster.model_dump_json()}\nEvidence:\n{evidence}\nRunbooks:\n{rb}", fallback)

class TimelineAgent(BaseIncidentAgent):
    def run(self, cluster: EvidenceCluster) -> IncidentTimeline:
        events = [TimelineEvent(timestamp=e.timestamp, source_file=e.source_file, line_no=e.line_no, service=e.service, level=e.level, message=e.message) for e in cluster.evidence]
        events = sorted(events, key=lambda e: (e.timestamp or datetime.min, e.line_no))[:20]
        interp = f"Timeline contains {len(events)} evidence point(s) for {cluster.candidate_category.value}, ordered by timestamp/line number."
        return IncidentTimeline(events=events, interpretation=interp)

class SymptomCauseAnalyzerAgent(BaseIncidentAgent):
    def run(self, cluster: EvidenceCluster) -> str:
        if cluster.candidate_category == IncidentCategory.API_TIMEOUT:
            return "API timeout may be a downstream symptom; inspect database, network, external dependency, or deployment signals in nearby clusters before treating it as root cause."
        if cluster.candidate_category == IncidentCategory.DEPLOYMENT_REGRESSION:
            return "Deployment correlation is meaningful but not automatically causal; compare new-version-only failures with prior baseline."
        return f"{cluster.signature} is treated as the strongest cause signal for this cluster, with other errors considered possible symptoms."

class RootCauseAnalysisAgent(BaseIncidentAgent):
    def run(self, cluster: EvidenceCluster, classification: Dict[str, Any], runbooks: RunbookRetrievalResult) -> RootCauseAnalysis:
        refs = [evidence_ref(e) for e in cluster.evidence[:8]]
        category = classification.get("category", cluster.candidate_category.value)
        fallback = {
            "primary_root_cause": f"Likely {category} incident indicated by {cluster.signature}: {cluster.summary}",
            "alternative_causes": ["Related dependency failure", "Recent deployment/config change", "Insufficient telemetry for full certainty"],
            "supporting_evidence": refs,
            "missing_evidence": ["Metrics around the affected services", "Recent deploy/config history", "Trace spans across dependencies"],
            "confidence": classification.get("confidence", 0.65),
        }
        evidence = "\n".join(refs)
        rb = "\n".join("; ".join(c.diagnostics + c.remediation) for c in runbooks.chunks[:2])
        data = self.llm_json("Return JSON matching primary_root_cause, alternative_causes, supporting_evidence, missing_evidence, confidence. Do not invent evidence.", f"Category={category}\nCluster={cluster.summary}\nEvidence:\n{evidence}\nRunbook hints:\n{rb}", fallback)
        return RootCauseAnalysis(**{**fallback, **data})

class RemediationAgent(BaseIncidentAgent):
    def run(self, cluster: EvidenceCluster, rca: RootCauseAnalysis, runbooks: RunbookRetrievalResult) -> RemediationPlan:
        actions=[]; validations=[]; safety=[]
        for rb in runbooks.chunks[:2]:
            actions.extend(rb.remediation[:3]); validations.extend(rb.validation[:3]); safety.extend(rb.safety_notes[:2])
        fallback = {
            "immediate_actions": list(dict.fromkeys(actions or ["Triage affected services", "Collect metrics/traces", "Prepare rollback or mitigation plan"])),
            "validation_checks": list(dict.fromkeys(validations or ["Error rate decreases", "Latency returns to baseline"])),
            "safety_notes": list(dict.fromkeys(safety + ["Preview only; human approval is required before external action."])),
            "requires_human_approval": True,
        }
        data = self.llm_json("Return JSON with immediate_actions, validation_checks, safety_notes, requires_human_approval. Keep actions safe.", f"RCA={rca.model_dump_json()}\nCluster={cluster.model_dump_json()}\nRunbooks={runbooks.model_dump_json()}", fallback)
        return RemediationPlan(**{**fallback, **data})

class EvidenceGroundingCriticAgent(BaseIncidentAgent):
    def run(self, cluster: EvidenceCluster, rca: RootCauseAnalysis, remediation: RemediationPlan) -> List[str]:
        findings=[]
        if not rca.supporting_evidence: findings.append("RCA has no supporting evidence references.")
        if not all(":" in x for x in rca.supporting_evidence[:3]): findings.append("Some evidence references may not include file:line format.")
        risky = [a for a in remediation.immediate_actions if re.search(r"delete|restart db|drop|truncate|disable auth", a, re.I)]
        if risky: findings.append("Risky remediation requires explicit human approval: " + "; ".join(risky[:3]))
        if cluster.candidate_category == IncidentCategory.API_TIMEOUT:
            findings.append("Check whether API timeout is a symptom of another upstream cluster before final escalation.")
        return findings or ["Evidence grounding and safety checks passed for preview mode."]

class HumanApprovalGateAgent:
    def run(self, severity: SeverityLevel) -> ApprovalStatus:
        return ApprovalStatus.PENDING_HUMAN_APPROVAL if severity in {SeverityLevel.P1, SeverityLevel.P2} else ApprovalStatus.PREVIEW_ONLY

class NotificationPreviewAgent:
    def run(self, cluster: EvidenceCluster, severity: SeverityLevel, rca: RootCauseAnalysis, approval: ApprovalStatus) -> NotificationPayload:
        title = f"{severity.value}: {cluster.candidate_category.value} incident candidate"
        body = f"Affected services: {', '.join(cluster.affected_services) or 'unknown'}\nLikely cause: {rca.primary_root_cause}\nTop evidence: {evidence_ref(cluster.evidence[0]) if cluster.evidence else 'none'}\nStatus: {approval.value}; no external message sent."
        return NotificationPayload(title=title, body=body, approval_status=approval)

class TicketPreviewAgent:
    def run(self, cluster: EvidenceCluster, severity: SeverityLevel, rca: RootCauseAnalysis, remediation: RemediationPlan, approval: ApprovalStatus) -> TicketPayload:
        title = f"{severity.value}: {cluster.candidate_category.value} - {cluster.signature}"
        desc = "\n".join(["Root cause: " + rca.primary_root_cause, "Evidence:", *["- "+x for x in rca.supporting_evidence[:8]], "Actions:", *["- "+x for x in remediation.immediate_actions[:6]]])
        return TicketPayload(title=title, priority=severity, labels=["ai-generated", "incident", severity.value, cluster.candidate_category.value.lower().replace("/","-")], description=desc, approval_status=approval)

class CookbookSynthesisAgent:
    def run(self, cluster: EvidenceCluster, remediation: RemediationPlan) -> CookbookChecklist:
        return CookbookChecklist(title=f"Cookbook: {cluster.candidate_category.value} / {cluster.signature}", steps=remediation.immediate_actions, validation=remediation.validation_checks, safety_notes=remediation.safety_notes)


## 9. `graph/` — LangGraph-compatible orchestration

Maps to:

```text
src/graph/state.py
src/graph/nodes.py
src/graph/edges.py
src/graph/builder.py
src/graph/runner.py
src/graph/fallbacks.py
src/graph/approval.py
```

The graph runs once over all clusters and aggregates incident results. If `langgraph` is unavailable, it uses the same node order deterministically.


In [24]:
class IncidentGraphRunner:
    def __init__(self, app_settings: AppSettings, llm_settings: LLMSettings):
        self.settings = app_settings
        self.runbook_retriever = RunbookRetriever(RunbookStore(RUNBOOKS))
        self.classifier = IncidentClassifierAgent(llm_settings, enabled=app_settings.enable_llm_mode)
        self.timeline = TimelineAgent(llm_settings, enabled=False)
        self.symptom = SymptomCauseAnalyzerAgent(llm_settings, enabled=app_settings.enable_llm_mode)
        self.rca = RootCauseAnalysisAgent(llm_settings, enabled=app_settings.enable_llm_mode)
        self.remediation = RemediationAgent(llm_settings, enabled=app_settings.enable_llm_mode)
        self.critic = EvidenceGroundingCriticAgent(llm_settings, enabled=app_settings.enable_llm_mode)
        self.approval = HumanApprovalGateAgent()
        self.notification = NotificationPreviewAgent()
        self.ticket = TicketPreviewAgent()
        self.cookbook = CookbookSynthesisAgent()
    def analyze_cluster(self, cluster: EvidenceCluster) -> IncidentAnalysisResult:
        runbooks = self.runbook_retriever.retrieve(cluster)
        classification = self.classifier.run(cluster, runbooks)
        category = IncidentCategory(classification.get("category", cluster.candidate_category.value)) if classification.get("category", cluster.candidate_category.value) in [c.value for c in IncidentCategory] else cluster.candidate_category
        severity = SeverityLevel(classification.get("severity", cluster.severity_hint.value)) if classification.get("severity", cluster.severity_hint.value) in [s.value for s in SeverityLevel] else cluster.severity_hint
        timeline = self.timeline.run(cluster)
        symptom_vs_cause = classification.get("symptom_vs_cause") or self.symptom.run(cluster)
        rca = self.rca.run(cluster, classification, runbooks)
        plan = self.remediation.run(cluster, rca, runbooks)
        findings = self.critic.run(cluster, rca, plan)
        approval = self.approval.run(severity)
        note = self.notification.run(cluster, severity, rca, approval)
        ticket = self.ticket.run(cluster, severity, rca, plan, approval)
        cookbook = self.cookbook.run(cluster, plan)
        return IncidentAnalysisResult(
            incident_id=cluster.cluster_id, cluster=cluster, category=category, severity=severity,
            confidence=float(classification.get("confidence", rca.confidence or 0.65)), symptom_vs_cause=symptom_vs_cause,
            timeline=timeline, rca=rca, remediation=plan, notification=note, ticket=ticket, cookbook=cookbook,
            critic_findings=findings, evidence_grounded=bool(rca.supporting_evidence)
        )
    def run(self, state: IncidentGraphState) -> IncidentGraphState:
        incidents=[]
        for c in tqdm(state.clusters, desc="Running agents"):
            try:
                incidents.append(self.analyze_cluster(c))
            except Exception as e:
                state.errors.append(f"Cluster {c.cluster_id} failed: {e}")
        state.incidents = incidents
        return state


## 10. `reporting/` — Markdown, JSON, CSV, notification/ticket previews, output bundle

Maps to:

```text
src/reporting/markdown_report.py
src/reporting/json_exporter.py
src/reporting/csv_exporter.py
src/reporting/notification_preview.py
src/reporting/ticket_preview.py
src/reporting/cookbook_exporter.py
src/reporting/export_bundle.py
```


In [25]:
def incident_to_flat_row(i: IncidentAnalysisResult) -> Dict[str, Any]:
    c=i.cluster
    return {
        "incident_id": i.incident_id, "category": i.category.value, "severity": i.severity.value, "confidence": round(i.confidence,3),
        "signature": c.signature, "source_file": c.evidence[0].source_file if c.evidence else "", "evidence_count": c.signal_count,
        "total_weight": round(c.total_weight,2), "services": "|".join(c.affected_services), "ticket_title": i.ticket.title,
        "approval_status": i.ticket.approval_status.value, "root_cause": i.rca.primary_root_cause,
    }

class MarkdownIncidentReportBuilder:
    def build(self, state: IncidentGraphState) -> str:
        lines=["# Incident Analysis Report", "", f"Raw files: {len(state.raw_files)}", f"Events parsed: {len(state.events)}", f"Signals: {len(state.signals)}", f"Clusters: {len(state.clusters)}", f"Incidents: {len(state.incidents)}", ""]
        if state.eval_summary:
            e=state.eval_summary
            lines += ["## Eval Summary", f"- Cases: {e.total_cases}", f"- Matched cases: {e.matched_cases}", f"- Category recall: {e.category_recall:.1%}", f"- Severity accuracy on matched: {e.severity_accuracy_on_matched:.1%}", f"- Evidence term hit rate: {e.evidence_term_hit_rate_on_matched:.1%}", ""]
        for i in state.incidents[:50]:
            lines += [f"## {i.severity.value} {i.category.value} — {i.cluster.signature}", "", f"Incident ID: `{i.incident_id}`", f"Affected services: {', '.join(i.cluster.affected_services) or 'unknown'}", f"Confidence: {i.confidence:.2f}", f"Symptom vs cause: {i.symptom_vs_cause}", "", "### RCA", i.rca.primary_root_cause, "", "### Evidence", *[f"- {x}" for x in i.rca.supporting_evidence[:8]], "", "### Remediation", *[f"- {x}" for x in i.remediation.immediate_actions[:8]], "", "### Validation", *[f"- {x}" for x in i.remediation.validation_checks[:6]], "", "### Preview Ticket", f"- {i.ticket.title}", f"- Approval: {i.ticket.approval_status.value}", ""]
        return "\n".join(lines)

class OutputBundleBuilder:
    def __init__(self, settings: AppSettings): self.settings=settings
    def export(self, state: IncidentGraphState) -> ExportManifest:
        run_dir = self.settings.output_dir / f"outputs_{stable_hash(str(datetime.now()))}"
        run_dir.mkdir(parents=True, exist_ok=True)
        md_text = MarkdownIncidentReportBuilder().build(state)
        md_path = run_dir / "incident_report.md"; md_path.write_text(md_text, encoding="utf-8")
        json_path = run_dir / "incident_results.json"; json_path.write_text(json.dumps([i.model_dump(mode="json") for i in state.incidents], indent=2, default=str), encoding="utf-8")
        csv_path = run_dir / "incident_results.csv"; pd.DataFrame([incident_to_flat_row(i) for i in state.incidents]).to_csv(csv_path, index=False)
        if state.eval_summary:
            (run_dir / "eval_summary.json").write_text(state.eval_summary.model_dump_json(indent=2), encoding="utf-8")
        zip_path = shutil.make_archive(str(run_dir), "zip", run_dir)
        state.exports = ExportManifest(markdown_report=str(md_path), json_export=str(json_path), csv_export=str(csv_path), output_zip=zip_path)
        return state.exports


## 11. `evals/` — hidden ground truth loader and scorer

Maps to:

```text
src/evals/ground_truth_loader.py
src/evals/scorer.py
src/evals/metrics.py
src/evals/scorecard.py
src/evals/report.py
```

This cell is the only place that reads `ground_truth_eval_only`, and it is called only after inference exists.


In [26]:
class GroundTruthLoader:
    def load(self, files: List[Path]) -> List[GroundTruthCase]:
        cases=[]
        for p in files:
            if p.name == "golden_labels.json":
                data=json.loads(p.read_text(encoding="utf-8"))
                for row in data:
                    if isinstance(row.get("affected_services"), str):
                        row["affected_services"] = row["affected_services"].split("|")
                    cases.append(GroundTruthCase(**row))
        return cases

class EvaluationScorer:
    def _incident_file(self, incident: IncidentAnalysisResult) -> str:
        return incident.cluster.evidence[0].source_file if incident.cluster.evidence else ""
    def _terms_hit(self, case: GroundTruthCase, incident: IncidentAnalysisResult) -> bool:
        terms = [t.lower() for t in flatten_terms(case.expected_evidence_terms) if len(str(t).strip()) >= 3]
        if not terms: return True
        text = " ".join([e.message for e in incident.cluster.evidence] + incident.rca.supporting_evidence).lower()
        # Good hit: one full phrase or at least two smaller terms.
        phrase_hits = sum(1 for t in terms if len(t.split()) >= 2 and t in text)
        token_hits = sum(1 for t in terms if len(t.split()) < 2 and t in text)
        return phrase_hits >= 1 or token_hits >= min(2, len(terms))
    def _ticket_expected(self, case: GroundTruthCase) -> bool:
        return bool(case.expected_behavior.get("create_ticket", case.expected_severity in ["P1","P2"]))
    def match_case(self, case: GroundTruthCase, incidents: List[IncidentAnalysisResult]) -> Optional[IncidentAnalysisResult]:
        raw_tail = case.raw_file.split("/")[-1]
        same_file = [i for i in incidents if self._incident_file(i).endswith(raw_tail)]
        same_cat = [i for i in same_file if i.category.value.lower() == case.expected_category.lower()]
        if not same_cat: return None
        # Prefer evidence-term hit, then service overlap, then highest confidence/weight.
        def score(i):
            svc_overlap=len(set(s.lower() for s in case.affected_services) & set(s.lower() for s in i.cluster.affected_services))
            return (self._terms_hit(case, i), svc_overlap, i.cluster.total_weight, i.confidence)
        return sorted(same_cat, key=score, reverse=True)[0]
    def score(self, cases: List[GroundTruthCase], incidents: List[IncidentAnalysisResult]) -> EvalSummary:
        results=[]
        for case in cases:
            inc=self.match_case(case, incidents)
            if not inc:
                results.append(EvalResult(case_id=case.case_id, matched_incident_id=None, category_match=False, severity_match=False, ticket_match=False, evidence_term_hit=False)); continue
            ticket_created = inc.ticket.priority in {SeverityLevel.P1, SeverityLevel.P2}
            results.append(EvalResult(
                case_id=case.case_id, matched_incident_id=inc.incident_id,
                category_match=inc.category.value.lower()==case.expected_category.lower(),
                severity_match=inc.severity.value==case.expected_severity,
                ticket_match=ticket_created==self._ticket_expected(case),
                evidence_term_hit=self._terms_hit(case, inc)
            ))
        matched=[r for r in results if r.matched_incident_id]
        denom=len(cases) or 1; mden=len(matched) or 1
        return EvalSummary(total_cases=len(cases), matched_cases=len(matched), category_recall=sum(r.category_match for r in results)/denom, severity_accuracy_on_matched=sum(r.severity_match for r in matched)/mden, ticket_trigger_accuracy_on_matched=sum(r.ticket_match for r in matched)/mden, evidence_term_hit_rate_on_matched=sum(r.evidence_term_hit for r in matched)/mden, results=results)


## 12. End-to-end pipeline runner

This is the notebook equivalent of the future `src/graph/runner.py` plus app entrypoint. Each prior cell remains independently refactorable.


In [27]:
def run_pipeline(input_paths: List[str], run_evals: bool = True, max_clusters: Optional[int] = None) -> IncidentGraphState:
    state = IncidentGraphState(input_paths=input_paths)
    ingestion = IngestionService(APP_SETTINGS)
    root, raw_files, gt_files, events = ingestion.run(input_paths)
    state.runtime_root = str(root)
    state.raw_files = [safe_relpath(p, root) for p in raw_files]
    state.ground_truth_files = [safe_relpath(p, root) for p in gt_files]
    state.events = events
    extractor = SignalExtractor(SignalRuleEngine(SIGNAL_RULES))
    state.signals = extractor.extract(events)
    clusterer = EvidenceClusterer(SeverityHeuristicEngine(), HybridClusterStrategy(APP_SETTINGS.default_time_window_minutes), APP_SETTINGS)
    state.clusters = IncidentDeduplicator().dedupe(clusterer.cluster(state.signals))
    if max_clusters:
        state.clusters = state.clusters[:max_clusters]
    print(f"Parsed events={len(state.events)} signals={len(state.signals)} clusters={len(state.clusters)}")
    graph = IncidentGraphRunner(APP_SETTINGS, LLM_SETTINGS)
    state = graph.run(state)
    if run_evals and APP_SETTINGS.enable_eval_mode and state.ground_truth_files:
        cases = GroundTruthLoader().load([Path(state.runtime_root) / p for p in state.ground_truth_files])
        state.eval_summary = EvaluationScorer().score(cases, state.incidents)
    OutputBundleBuilder(APP_SETTINGS).export(state)
    return state

def show_run_summary(state: IncidentGraphState):
    print("Raw files:", len(state.raw_files))
    print("Events:", len(state.events), "Signals:", len(state.signals), "Clusters:", len(state.clusters), "Incidents:", len(state.incidents))
    if state.eval_summary:
        e=state.eval_summary
        print(f"Eval: matched {e.matched_cases}/{e.total_cases}; category recall={e.category_recall:.1%}; severity accuracy={e.severity_accuracy_on_matched:.1%}; evidence hit={e.evidence_term_hit_rate_on_matched:.1%}")
    print("Exports:", state.exports.model_dump())


## 13. Colab batch execution layer

This is intentionally placed above the upload/smoke-test cells. In the future `src` project it maps to `graph/runner.py`, `graph/checkpointing.py`, and a thin app/controller layer.

The batch layer is optimized for Colab:

- deterministic parsing, normalization, signal extraction, and clustering happen before agent reasoning
- each ZIP/log path runs in an isolated output directory
- checkpoint metadata allows quick resume/skip behavior
- OpenRouter/LLM mode remains optional and controlled by environment settings
- hidden ground truth is still loaded only by the eval step after inference


In [28]:
@dataclass
class BatchRunConfig:
    input_paths: List[str]
    run_evals: bool = True
    max_clusters_per_input: Optional[int] = None
    continue_on_error: bool = True
    resume_completed: bool = True
    batch_name: str = field(default_factory=lambda: f"batch_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}")

@dataclass
class BatchRunRecord:
    input_path: str
    status: str
    started_at: str
    finished_at: str
    output_dir: Optional[str] = None
    error: Optional[str] = None
    events: int = 0
    signals: int = 0
    clusters: int = 0
    incidents: int = 0
    eval_total_cases: int = 0
    eval_matched_cases: int = 0
    category_recall: float = 0.0
    severity_accuracy: float = 0.0
    evidence_hit_rate: float = 0.0

@dataclass
class BatchExecutionSummary:
    batch_name: str
    records: List[BatchRunRecord]
    manifest_path: str
    def print_summary(self):
        ok = sum(1 for r in self.records if r.status == "completed")
        failed = sum(1 for r in self.records if r.status == "failed")
        skipped = sum(1 for r in self.records if r.status == "skipped")
        print(f"Batch {self.batch_name}: completed={ok}, failed={failed}, skipped={skipped}")
        for r in self.records:
            print(f"- {r.status}: {Path(r.input_path).name} | incidents={r.incidents} | eval={r.eval_matched_cases}/{r.eval_total_cases} | severity={r.severity_accuracy:.1%}")
        print("Manifest:", self.manifest_path)

class ColabBatchExecutor:
    """Notebook app/controller layer for batch execution.

    This class intentionally does orchestration only. Domain work remains in the
    src-aligned classes above: ingest, signals, clustering, graph, reporting, evals.
    """
    def __init__(self, settings: AppSettings):
        self.settings = settings
    def _batch_root(self, batch_name: str) -> Path:
        root = self.settings.output_dir / batch_name
        root.mkdir(parents=True, exist_ok=True)
        return root
    def _record_path(self, batch_root: Path, input_path: str) -> Path:
        return batch_root / f"{stable_id(input_path, length=12)}_record.json"
    def _manifest_path(self, batch_root: Path) -> Path:
        return batch_root / "batch_manifest.json"
    def _is_completed(self, batch_root: Path, input_path: str) -> bool:
        p = self._record_path(batch_root, input_path)
        if not p.exists():
            return False
        try:
            return json.loads(p.read_text()).get("status") == "completed"
        except Exception:
            return False
    def discover_inputs(self, roots: List[str]) -> List[str]:
        discovered=[]
        supported={".zip", ".jsonl", ".log", ".txt"}
        for raw in roots:
            p=Path(raw)
            if p.is_file() and p.suffix.lower() in supported:
                discovered.append(str(p))
            elif p.is_dir():
                for child in sorted(p.rglob("*")):
                    if "ground_truth_eval_only" in child.parts:
                        continue
                    if child.is_file() and child.suffix.lower() in supported:
                        discovered.append(str(child))
        # Stable de-dupe preserves order.
        seen=set(); result=[]
        for item in discovered:
            if item not in seen:
                seen.add(item); result.append(item)
        return result
    def _make_record(self, input_path: str, status: str, started: str, finished: str, state: Optional[IncidentGraphState] = None, error: Optional[str] = None) -> BatchRunRecord:
        rec=BatchRunRecord(input_path=input_path, status=status, started_at=started, finished_at=finished, error=error)
        if state:
            rec.output_dir = str(self.settings.output_dir)
            rec.events=len(state.events); rec.signals=len(state.signals); rec.clusters=len(state.clusters); rec.incidents=len(state.incidents)
            if state.eval_summary:
                rec.eval_total_cases=state.eval_summary.total_cases
                rec.eval_matched_cases=state.eval_summary.matched_cases
                rec.category_recall=state.eval_summary.category_recall
                rec.severity_accuracy=state.eval_summary.severity_accuracy_on_matched
                rec.evidence_hit_rate=state.eval_summary.evidence_term_hit_rate_on_matched
        return rec
    def run(self, config: BatchRunConfig) -> BatchExecutionSummary:
        batch_root=self._batch_root(config.batch_name)
        records=[]
        input_paths=self.discover_inputs(config.input_paths)
        if not input_paths:
            raise ValueError("No supported input files discovered. Provide .zip, .jsonl, .log, .txt, or a folder containing them.")
        for input_path in input_paths:
            rec_path=self._record_path(batch_root, input_path)
            if config.resume_completed and self._is_completed(batch_root, input_path):
                data=json.loads(rec_path.read_text())
                records.append(BatchRunRecord(**data))
                records[-1].status="skipped"
                continue
            started=datetime.now(timezone.utc).isoformat()
            try:
                print(f"\n=== Running {Path(input_path).name} ===")
                state=run_pipeline([input_path], run_evals=config.run_evals, max_clusters=config.max_clusters_per_input)
                finished=datetime.now(timezone.utc).isoformat()
                rec=self._make_record(input_path, "completed", started, finished, state=state)
                rec_path.write_text(json.dumps(asdict(rec), indent=2), encoding="utf-8")
                records.append(rec)
            except Exception as exc:
                finished=datetime.now(timezone.utc).isoformat()
                rec=self._make_record(input_path, "failed", started, finished, error=repr(exc))
                rec_path.write_text(json.dumps(asdict(rec), indent=2), encoding="utf-8")
                records.append(rec)
                if not config.continue_on_error:
                    raise
        manifest=self._manifest_path(batch_root)
        payload={"batch_name": config.batch_name, "records": [asdict(r) for r in records]}
        manifest.write_text(json.dumps(payload, indent=2), encoding="utf-8")
        summary=BatchExecutionSummary(config.batch_name, records, str(manifest))
        summary.print_summary()
        return summary

def run_colab_batch(input_paths: List[str], run_evals: bool = True, max_clusters_per_input: Optional[int] = None, batch_name: Optional[str] = None) -> BatchExecutionSummary:
    config = BatchRunConfig(input_paths=input_paths, run_evals=run_evals, max_clusters_per_input=max_clusters_per_input)
    if batch_name:
        config.batch_name = batch_name
    return ColabBatchExecutor(APP_SETTINGS).run(config)


## 14. Colab upload cell

Use this in Colab if you want to upload a ZIP/log file manually.


In [36]:
from google.colab import files
uploaded = files.upload()
uploaded_paths = []
for name, data in uploaded.items():
    dest = APP_SETTINGS.upload_dir / name
    dest.write_bytes(data)
    uploaded_paths.append(str(dest))
# To save tokens, you can limit the number of clusters processed by the LLM:
# state = run_pipeline(uploaded_paths, run_evals=True, max_clusters=10)
# Or disable LLM mode entirely:
# APP_SETTINGS.enable_llm_mode = False # Add this line before calling run_pipeline if you want to disable LLM
# To run the full set without limiting clusters, remove the 'max_clusters' argument or set it to None:
state = run_pipeline(uploaded_paths, run_evals=True, max_clusters=None)
show_run_summary(state)

Saving devops_raw_log_corpus_v2_all.zip to devops_raw_log_corpus_v2_all (2).zip
Extracted devops_raw_log_corpus_v2_all (2).zip -> /content/incidentiq_workspace/extracted/run_33ef691f1924/devops_raw_log_corpus_v2_all (2)


Parsing logs:   0%|          | 0/14 [00:00<?, ?it/s]

Extracting signals:   0%|          | 0/13063 [00:00<?, ?it/s]

Parsed events=13063 signals=3598 clusters=314
API Key Found. We are good to go...
API Key Found. We are good to go...
API Key Found. We are good to go...
API Key Found. We are good to go...
API Key Found. We are good to go...


Running agents:   0%|          | 0/314 [00:00<?, ?it/s]

Raw files: 14
Events: 13063 Signals: 3598 Clusters: 314 Incidents: 0
Eval: matched 0/42; category recall=0.0%; severity accuracy=0.0%; evidence hit=0.0%
Exports: {'markdown_report': '/content/incidentiq_workspace/outputs/outputs_b78362f8fa65/incident_report.md', 'json_export': '/content/incidentiq_workspace/outputs/outputs_b78362f8fa65/incident_results.json', 'csv_export': '/content/incidentiq_workspace/outputs/outputs_b78362f8fa65/incident_results.csv', 'output_zip': '/content/incidentiq_workspace/outputs/outputs_b78362f8fa65.zip'}


In [37]:
if 'state' in globals() and state.eval_summary:
    print("Detailed Evaluation Results:")
    display(pd.DataFrame([r.model_dump() for r in state.eval_summary.results]))
else:
    print("Run the pipeline first to get eval_summary.")

Detailed Evaluation Results:


,case_id,matched_incident_id,category_match,severity_match,ticket_match,evidence_term_hit
0,DB_POOL,None,False,False,False,False
1,AUTH_JWT,None,False,False,False,False
2,API_504,None,False,False,False,False
3,MEM_OOM,None,False,False,False,False
4,QUEUE_LAG,None,False,False,False,False
5,DISK_FULL,None,False,False,False,False
6,DEPLOY_ERROR,None,False,False,False,False
7,EXT_PAY,None,False,False,False,False
8,UNK_STATE,None,False,False,False,False
9,NET_RESET,None,False,False,False,False


In [ ]:
if 'state' in globals() and state.eval_summary:
    # Load ground truth cases to get expected categories
    gt_loader = GroundTruthLoader()
    gt_cases = gt_loader.load([Path(state.runtime_root) / p for p in state.ground_truth_files])
    gt_case_map = {c.case_id: c for c in gt_cases}

    # Map incidents by their incident_id for easy lookup
    incident_map = {inc.incident_id: inc for inc in state.incidents}

    print("\n--- Detailed Category Mismatch Analysis ---")
    mismatched_found = False
    for eval_result in state.eval_summary.results:
        if eval_result.matched_incident_id and not eval_result.category_match:
            mismatched_found = True
            gt_case = gt_case_map.get(eval_result.case_id)
            matched_incident = incident_map.get(eval_result.matched_incident_id)

            if gt_case and matched_incident:
                print(f"Case ID: {eval_result.case_id}")
                print(f"  Expected Category: {gt_case.expected_category}")
                print(f"  Predicted Category: {matched_incident.category.value}")
                print(f"  Incident ID: {matched_incident.incident_id}")
                print(f"  Incident Signature: {matched_incident.cluster.signature}")
                print(f"  Top Evidence: {evidence_ref(matched_incident.cluster.evidence[0]) if matched_incident.cluster.evidence else 'N/A'}")
                print("------------------------------------------")
            else:
                print(f"Warning: Could not find ground truth case or incident for mismatch: {eval_result.case_id}")

    if not mismatched_found:
        print("No category mismatches found among matched incidents.")
        if state.eval_summary.matched_cases == 0:
            print("However, no incidents were matched to ground truth cases at all. Check `matched_incident_id` for None values.")
    print("-----------------------------------------")

else:
    print("Run the pipeline and evaluation first to get detailed results.")

In [33]:
# To Check if the key is working and you are getting the output, uncomment and run the following code.
# class LLMTestAgent(BaseIncidentAgent):
#     def test_call(self, prompt: str) -> Dict[str, Any]:
#         # Use a very small fallback to ensure LLM is actually called if enabled
#         return self.llm_json("Return only JSON with a 'response' key.", prompt, {"response": "LLM disabled or failed."})

# # Instantiate the test agent
# test_agent = LLMTestAgent(LLM_SETTINGS, enabled=True)

# # Make a small test call if the LLM is available
# if test_agent.enabled:
#     print("Attempting a test LLM call...")
#     test_output = test_agent.test_call("Say hello in a JSON object with key 'response'.")
#     print("Test LLM output:", test_output)
#     if "hello" in str(test_output).lower():
#         print("LLM call successful!")
#     else:
#         print("LLM call returned unexpected output.")
# else:
#     print("LLM is not enabled or API key is missing/invalid, skipping test call.")

API Key Found. We are good to go...
Attempting a test LLM call...
Test LLM output: {'response': 'Hello!'}
LLM call successful!


## 15. Local / attached sample dataset smoke test

In this environment the sample corpus is already present at `/mnt/data/devops_raw_log_corpus_v2_all.zip`. In Colab, change the path to your uploaded ZIP.


In [38]:
DEFAULT_LOCAL_DATASET = Path("/mnt/data/devops_raw_log_corpus_v2_all.zip")
if DEFAULT_LOCAL_DATASET.exists():
    state = run_pipeline([str(DEFAULT_LOCAL_DATASET)], run_evals=True)
    show_run_summary(state)
else:
    print("Default local dataset not found. Use the upload cell above.")


Default local dataset not found. Use the upload cell above.


## 15b. Optional batch smoke test

Use this when you have several uploaded ZIP/log files or a folder of datasets. It runs each input independently and writes checkpoint records under the output directory.


In [ ]:
# Example:
# batch_summary = run_colab_batch([str(DEFAULT_LOCAL_DATASET)], run_evals=True, batch_name="quick_batch_test")
# batch_summary.print_summary()


## 16. Inspect incidents and eval details


In [ ]:
if "state" in globals():
    display(pd.DataFrame([incident_to_flat_row(i) for i in state.incidents]).head(50))
    if state.eval_summary:
        display(pd.DataFrame([r.model_dump() for r in state.eval_summary.results]))
else:
    print("Run the pipeline first.")


## 17. Preview report and download output bundle


In [ ]:
if "state" in globals() and state.exports.markdown_report:
    report_text = Path(state.exports.markdown_report).read_text(encoding="utf-8")
    print(report_text[:6000])
    print("\nOutput ZIP:", state.exports.output_zip)
    # In Colab:
    # from google.colab import files
    # files.download(state.exports.output_zip)
else:
    print("Run the pipeline first.")


## 18. Refactor checklist to `src/`

Split the cells exactly into the README-aligned structure:

```text
src/config/settings.py                 AppSettings
src/config/llm_settings.py             LLMSettings
src/models/*.py                        Pydantic contracts/enums
src/utils/*.py                         safe paths, time, JSON/text/hash helpers
src/ingest/*.py                        InputValidator, SafeZipLoader, discovery, parser, normalizer
src/signals/*.py                       SIGNAL_RULES, SignalRuleEngine, SignalExtractor, severity/category heuristics
src/clustering/*.py                    HybridClusterStrategy, EvidenceClusterer, IncidentDeduplicator
src/rag/*.py                           RunbookStore, RetrievalQueryBuilder, RunbookRetriever
src/agents/*.py                        Focused agents with structured outputs
src/graph/*.py                         IncidentGraphState, node order, runner, approval routing
src/reporting/*.py                     Markdown/JSON/CSV/export bundle builders
src/evals/*.py                         GroundTruthLoader, EvaluationScorer, metrics/scorecard
```

Do not reintroduce `pipelines/` or `parsing/`; the README explicitly moved those responsibilities into `graph/`, `models/`, `ingest/`, and `signals/`.
